In [2]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [3]:
import copy
import optuna
import numpy as np
from transformers import AutoTokenizer

from src.train import train_single_fold, tokenize_df
from src.config import load_config
from src.data_holdout import get_pooled_df, get_fold_from_disk

import yaml
import torch
from datasets import Dataset
from pyprojroot import here
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import f1_score

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_name = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

In [11]:
config = load_config("configs/pubmedbert_frozen.yaml")

In [3]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

In [9]:

# open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

#choose model
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

#load full dataset and get folds
full_df = get_pooled_df()
train_folds, val_folds = get_fold_from_disk(full_df)

#load tokenizer and create tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

# tokenize training and val data
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

#function to freeze base model except for classification layer
def freeze_base(model):
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

#define optuna objective function
def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.2)
    num_epochs = trial.suggest_int("num_train_epochs", 5, 30)

    f1_scores = []

    #loop through folds
    for fold in range(5):
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3
        )
        freeze_base(model)

        training_args = TrainingArguments(
            output_dir=str(here(f"results/optuna/pubmedbert_frozen/trial-{trial.number}/fold-{fold}")),
            eval_strategy=base_data["eval_strategy"],
            save_strategy=base_data["save_strategy"],
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            num_train_epochs=num_epochs,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=base_data["greater_is_better"],
            fp16=base_data["fp16"],
            seed=base_data["seed"],
            report_to="none",
            save_total_limit=1,
            skip_memory_metrics=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
        )

        trainer.train()
        eval_results = trainer.evaluate()
        f1_scores.append(eval_results["eval_macro_f1"])

        #prevent memory build up
        del trainer
        del model
        torch.cuda.empty_cache()

    return float(np.mean(f1_scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best macro_f1:", study.best_trial.value)
print("Best params:", study.best_trial.params)

/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 12907.05 examples/s]
[I 2026-03-22 03:38:41,392] A new study created in memory with name: no-name-e0c6cde9-089c-43cc-876d-12f38cab02d7
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1783.49it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.d

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,1.013386,0.233333
2,No log,0.977053,0.246032
3,No log,0.974865,0.248521
4,No log,0.973941,0.248521
5,No log,0.969188,0.248521
6,No log,0.969231,0.248521
7,No log,0.965526,0.250980
8,No log,0.962974,0.250980
9,No log,0.960320,0.250980
10,No log,0.961078,0.250980


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.70it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4246.93it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                   

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,1.111655,0.142994
2,No log,0.974439,0.289744
3,No log,0.920905,0.253411
4,No log,0.923955,0.253411
5,No log,0.923324,0.253411
6,No log,0.924604,0.253411
7,No log,0.924581,0.253411


...

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3496.35it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                   

Epoch,Training Loss,Validation Loss,Macro F1
1,No log,2.103819,0.253411
2,No log,3.028487,0.337879
3,No log,1.509575,0.373115
4,No log,3.837292,0.284747
5,No log,2.904776,0.323917
6,No log,3.220185,0.317428
7,No log,1.815457,0.408466
8,No log,2.431611,0.307076
9,No log,2.044676,0.293400
10,No log,2.876040,0.315375


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.04it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

[I 2026-03-22 04:07:27,223] Trial 19 finished with value: 0.4166358677370642 and parameters: {'learning_rate': 0.09233718822470412, 'per_device_train_batch_size': 32, 'weight_decay': 0.03674371222818942, 'warmup_ratio': 0.08137869132973478, 'num_train_epochs': 25}. Best is trial 11 with value: 0.4331451163209793.


Best macro_f1: 0.4331451163209793
Best params: {'learning_rate': 0.05659457093822265, 'per_device_train_batch_size': 8, 'weight_decay': 0.09849942738842452, 'warmup_ratio': 0.1994321771543907, 'num_train_epochs': 16}


## Closer matching protocols

In [4]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

In [5]:
#open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

#get df
full_df = get_pooled_df()

# Same pattern as collaborator: all folds
train_folds, val_folds = get_fold_from_disk(full_df)

#load tokenizer and write tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

#tokenize all folds
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

for i in range(len(train_ds)):
    print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")

# define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

#Freeze encoder, train classifier head only
def freeze_base(model):
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False

#Set up Optuna objective
def objective(trial):
    # Match collaborator's structure, but use a frozen-head-appropriate LR range
    lr = trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True) #higher LR for classification head only
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.2) #empirically found that higher is better for classification head only
    num_epochs = trial.suggest_int("num_train_epochs", 12, 30) #not as many epochs needed for classification head only

    f1_scores = []

    for fold in range(5):
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3
        )
        freeze_base(model)

        training_args = TrainingArguments(
            output_dir=str(here(f"results/optuna/pubmedbert_frozen/trial-{trial.number}/fold-{fold}")),
            eval_strategy=base_data["eval_strategy"],
            save_strategy=base_data["save_strategy"],
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            num_train_epochs=num_epochs,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=base_data["greater_is_better"],
            fp16=base_data["fp16"],
            seed=base_data["seed"],
            logging_steps=10,
            report_to=base_data["report_to"],
            save_total_limit=1,
            skip_memory_metrics=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        trainer.train()
        f1_scores.append(trainer.evaluate()["eval_macro_f1"])

        del model
        torch.cuda.empty_cache()

    return float(np.mean(f1_scores))

# Study definition
db_path = here("results/optuna/optuna_study_pubmedbert_frozen.db")
storage_url = f"sqlite:///{db_path}"
n_startup_trials = 5

print(storage_url)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=n_startup_trials,
        seed=42,
    ),
    study_name="pubmedbert-frozen-hyperparameter-search2",
    storage=storage_url,
    load_if_exists = True
)

study.optimize(objective, n_trials=50)

# Best trials
print(f"\n{'='*60}")
print(f"  Best macro_f1: {study.best_trial.value:.4f}")
print(f"  Best params:   {study.best_trial.params}")
print(f"{'='*60}")


/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 9863.91 examples/s]


Fold 0: train=424 val=106
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106
sqlite:////workspace/exaggeration-detection/results/optuna/optuna_study_pubmedbert_frozen.db


[I 2026-03-22 05:19:39,445] Using an existing study with name 'pubmedbert-frozen-hyperparameter-search2' instead of creating a new one.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1785.18it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationsh

Epoch,Training Loss,Validation Loss,Macro F1
1,1.145472,1.716089,0.279609
2,1.191917,1.686313,0.291031
3,1.388109,1.763079,0.315789
4,1.300429,1.700441,0.281367
5,1.391123,1.430145,0.300292
6,1.282777,1.118647,0.405222
7,0.848788,1.207654,0.404466
8,1.349109,1.636893,0.372364
9,1.022690,1.618347,0.337627


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.26it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.55it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer

[I 2026-03-22 05:49:55,639] Trial 61 finished with value: 0.3823863400873394 and parameters: {'learning_rate': 0.0982767461145735, 'per_device_train_batch_size': 8, 'weight_decay': 0.06630763793423528, 'warmup_ratio': 0.08742640389667015, 'num_train_epochs': 22}. Best is trial 53 with value: 0.430736038423243.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1691.98it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.dec

Epoch,Training Loss,Validation Loss,Macro F1
1,1.026186,1.436165,0.282680
2,1.080855,1.078374,0.361277
3,1.018143,1.196764,0.368944
4,1.065933,1.260154,0.314738
5,1.055231,1.070317,0.288858
6,0.973519,1.034742,0.375306
7,0.810169,1.185618,0.310885
8,0.848408,1.096834,0.380862
9,0.756684,1.157416,0.311497
10,0.742452,1.039767,0.384263


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.52it/s]
[W 2026-03-22 05:50:17,370] Trial 62 failed with parameters: {'learning_rate': 0.0080796333895609, 'per_device_train_batch_size': 8, 'weight_decay': 0.029963208912724446, 'warmup_ratio': 0.06399504177584975, 'num_train_epochs': 13} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_104044/3923329275.py", line 99, in objective
    trainer.train()
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 1412, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", line 1865, in _inner_training_loop
    self._load_best_model()
  File "/usr/local/lib/python3.12/dist-packages/transformers/

KeyboardInterrupt: 

In [7]:
print(study.best_trial)
print(study.best_value)
print(study.best_params)

FrozenTrial(number=53, state=<TrialState.COMPLETE: 1>, values=[0.430736038423243], datetime_start=datetime.datetime(2026, 3, 22, 5, 39, 21, 883488), datetime_complete=datetime.datetime(2026, 3, 22, 5, 40, 38, 948178), params={'learning_rate': 0.006702663997044904, 'per_device_train_batch_size': 8, 'weight_decay': 0.036536591969597465, 'warmup_ratio': 0.05747520780203451, 'num_train_epochs': 14}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.1, log=True, low=0.001, step=None), 'per_device_train_batch_size': CategoricalDistribution(choices=(8,)), 'weight_decay': FloatDistribution(high=0.1, log=False, low=0.0, step=None), 'warmup_ratio': FloatDistribution(high=0.2, log=False, low=0.05, step=None), 'num_train_epochs': IntDistribution(high=30, log=False, low=12, step=1)}, trial_id=54, value=None)
0.430736038423243
{'learning_rate': 0.006702663997044904, 'per_device_train_batch_size': 8, 'weight_decay': 0.036536591969597465, 

In [8]:
first_50_trials = [t for t in study.trials if t.number < 50 and t.state == optuna.trial.TrialState.COMPLETE]

best_first_50 = max(first_50_trials, key=lambda t: t.value)

print("Best trial (first 50):")
print("Trial number:", best_first_50.number)
print("Value (macro_f1):", best_first_50.value)
print("Params:", best_first_50.params)

Best trial (first 50):
Trial number: 34
Value (macro_f1): 0.4275443204867935
Params: {'learning_rate': 0.015156709252430034, 'per_device_train_batch_size': 8, 'weight_decay': 0.06897032912868373, 'warmup_ratio': 0.05425597330104267, 'num_train_epochs': 23}
